In [9]:
# Cell to install and load packages only
%pip install python-dotenv
%pip install requests
%pip install openai
%pip install ipython
%pip install beautifulsoup4
%pip install lxml
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


... (4 outputs left out)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Imports

In [10]:
import os
import re
import json
from email.utils import parsedate_to_datetime

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# API Key Validation

In [11]:
load_dotenv()
openrouter_api_key = os.getenv("OPENROUTER_API_KEY").strip()

if openrouter_api_key.startswith("sk-or-v1-"):
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:9]}")
else:
    print("This does not look like an OpenRouter API key.")

OpenRouter API key exists and begins sk-or-v1-


In [12]:
problems = []
if openrouter_api_key.count("sk-or-") > 1:
    problems.append("It looks like it was pasted more than once")
for position, character in enumerate(openrouter_api_key, start=1):
    if not (character.isascii() and (character.isalnum() or character in "-_")):
        problems.append(f"Character {position} is {character!r}")

if problems:
    print("There is a problem with your key.")
    for problem in problems:
        print(f"- {problem}")
else:
    print(f"OpenRouter API key looks good and has {len(openrouter_api_key)} characters")

OpenRouter API key looks good and has 73 characters


# Check Connectivity

In [13]:
response = requests.get("https://openrouter.ai/api/v1/key", headers={"Authorization": f"Bearer {openrouter_api_key}"})
result = response.json()

if response.ok:
    key_info = result["data"]
    print(f"Success! OpenRouter accepted your key {key_info['label']}")
else:
    print(f"OpenRouter did not accept this key: {result['error']['message']} (error {response.status_code})")

Success! OpenRouter accepted your key sk-or-v1-398...d14


# Connect to OpenRouter

In [14]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openrouter/free"

# Retry up to 5 times when a free model is busy
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL, timeout=120, max_retries=5)

# Fetch the RSS Feed

In [15]:
RSS_URL = "https://nyaa.si/?page=rss&u=Yameii"
HEADERS = {"User-Agent": "Mozilla/5.0 (rss-reader)"}

feed_response = requests.get(RSS_URL, headers=HEADERS, timeout=30)
feed_response.raise_for_status()
print(f"Fetched {len(feed_response.content):,} bytes ({feed_response.headers.get('content-type')})")

Fetched 83,718 bytes (application/xml)


# Parse the XML

The `"xml"` parser (backed by `lxml`) keeps namespaced tags such as `<nyaa:seeders>` intact.

In [16]:
soup = BeautifulSoup(feed_response.content, "xml")

channel = soup.find("channel")
items = soup.find_all("item")
print(f"Channel: {channel.find('title').get_text(strip=True)}")
print(f"Found {len(items)} items")

Channel: Nyaa - Home - Torrent File RSS
Found 75 items


# Extract Fields

Release titles look like `[Yameii] Show Name - S02E04 v2 [English Dub] [CR WEB-DL 1080p H264 AAC] [CRC32] (Japanese Title)`, so the show name, season, episode, resolution and alternate title are pulled out with regexes.

In [17]:
TITLE_PATTERN = re.compile(r"^\[[^\]]+\]\s*(?P<show>.+?)\s+-\s+S(?P<season>\d+)E(?P<episode>\d+)")
RESOLUTION_PATTERN = re.compile(r"(\d{3,4}p)")
ALT_TITLE_PATTERN = re.compile(r"\]\s*\(([^)]*)\)\s*$")


def text_of(item, tag):
    """Return the stripped text of a child tag, or None if it is missing."""
    node = item.find(tag)
    return node.get_text(strip=True) if node else None


def to_int(value):
    return int(value) if value and value.isdigit() else None


def parse_title(title):
    match = TITLE_PATTERN.match(title)
    resolution = RESOLUTION_PATTERN.search(title)
    alt_title = ALT_TITLE_PATTERN.search(title)
    return {
        "show": match["show"] if match else title,
        "season": int(match["season"]) if match else None,
        "episode": int(match["episode"]) if match else None,
        "resolution": resolution.group(1) if resolution else None,
        "alt_title": alt_title.group(1) if alt_title else None,
    }


def parse_item(item):
    title = text_of(item, "title")
    pub_date = text_of(item, "pubDate")
    return {
        "title": title,
        **parse_title(title),
        "published": parsedate_to_datetime(pub_date) if pub_date else None,
        "size": text_of(item, "nyaa:size"),
        "category": text_of(item, "nyaa:category"),
        "seeders": to_int(text_of(item, "nyaa:seeders")),
        "leechers": to_int(text_of(item, "nyaa:leechers")),
        "downloads": to_int(text_of(item, "nyaa:downloads")),
        "comments": to_int(text_of(item, "nyaa:comments")),
        "trusted": text_of(item, "nyaa:trusted") == "Yes",
        "remake": text_of(item, "nyaa:remake") == "Yes",
        "info_hash": text_of(item, "nyaa:infoHash"),
        "view_url": text_of(item, "guid"),
        "torrent_url": text_of(item, "link"),
    }


records = [parse_item(item) for item in items]
records[0]

{'title': '[Yameii] Firefly Wedding - S01E04 v2 [English Dub] [CR WEB-DL 1080p H264 AAC] [A8BEB32F] (Hotaru no Yomeiri)',
 'show': 'Firefly Wedding',
 'season': 1,
 'episode': 4,
 'resolution': '1080p',
 'alt_title': 'Hotaru no Yomeiri',
 'published': datetime.datetime(2026, 11, 13, 20, 47, 48),
 'size': '1.3 GiB',
 'category': 'Anime - English-translated',
 'seeders': 90,
 'leechers': 6,
 'downloads': 353,
 'comments': 0,
 'trusted': False,
 'remake': False,
 'info_hash': '42ad5c597f08b3890e22cd3dac478bcc3909e65b',
 'view_url': 'https://nyaa.si/view/2185805',
 'torrent_url': 'https://nyaa.si/download/2185805.torrent'}

# View as a Table

In [18]:
df = pd.DataFrame(records)
display(df[["published", "show", "season", "episode", "resolution", "size", "seeders", "downloads"]])

,published,show,season,episode,resolution,size,seeders,downloads
0,2026-11-13 20:47:48,Firefly Wedding,1,4,1080p,1.3 GiB,90,353
1,2026-11-13 20:47:23,The Apothecary Diaries,3,5,1080p,1.4 GiB,443,1454
2,2026-11-13 20:44:55,Firefly Wedding,1,4,720p,685.2 MiB,40,161
3,2026-11-13 20:44:52,The Apothecary Diaries,3,5,720p,719.2 MiB,85,310
4,2026-11-13 04:08:07,Link Click,4,12,720p,669.4 MiB,34,213
...,...,...,...,...,...,...,...,...
70,2026-11-04 16:39:27,Magic Knight Rayearth (2026),1,3,1080p,1.3 GiB,30,451
71,2026-11-04 16:33:33,Magic Knight Rayearth (2026),1,3,720p,692.3 MiB,23,403
72,2026-11-04 14:42:23,The Detective Is Already Dead,2,3,1080p,1.4 GiB,19,329
73,2026-11-04 14:42:19,Saga of Tanya the Evil,2,9,1080p,1.4 GiB,65,892


# Unique Shows

Each episode is released in both 720p and 1080p, so collapse to one row per show before asking the LLM.

In [19]:
shows = (
    df.groupby("show")
    .agg(alt_title=("alt_title", "first"), releases=("title", "count"), downloads=("downloads", "sum"))
    .sort_values("downloads", ascending=False)
    .reset_index()
)
print(f"{len(shows)} unique shows")
shows

26 unique shows


,show,alt_title,releases,downloads
0,The Apothecary Diaries,Kusuriya no Hitorigoto Season 3 | S3,4,3732
1,The Exiled Heavy Knight Knows How to Game the ...,Tsuihou Sareta Tensei Juu Kishi wa Game Chishi...,4,3205
2,A Returner's Magic Should Be Special,Kikansha no Mahou wa Tokubetsu Desu Season 2 | S2,4,2563
3,Magical Explorer,Magical★Explorer: Eroge no Yuujin Chara ni Ten...,4,2394
4,Saga of Tanya the Evil,Youjo Senki II | Season 2 | S2,3,2096
5,The Iceblade Sorcerer Shall Rule the World,Hyouken no Majutsushi ga Sekai o Suberu II | S...,4,2034
6,Overgeared,Tempal: Item no Chikara,2,1974
7,Black Clover,NaN,2,1634
8,Magic Knight Rayearth (2026),NaN,4,1568
9,Link Click,Shiguang Dailiren III | Season 3 | S3 | Season...,4,1449


# Categorize by Season with the LLM

A simple prompt that asks the model to group the shows by season number.

In [32]:
season_lines = [
    f"- {row.show} (Season {row.season})"
    for row in df[["show", "season"]].drop_duplicates().itertuples()
]

system_prompt = "You are a helpful assistant. Group anime by their season number. Reply in markdown with one heading per season, listing the shows under each."
user_prompt = "Categorize these anime by season number:\n" + "\n".join(season_lines)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

response = openrouter.chat.completions.create(model=MODEL, messages=messages)
print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by baidu/ernie-5.5-flash-preview:free


## Season 1
- Firefly Wedding
- Dragon Ball Super: Beerus
- The Exiled Heavy Knight Knows How to Game the System
- FX Fighter Kurumi-chan
- Magic Knight Rayearth (2026)
- The World's Strongest Witch
- The Laid-Off Cheat-Granting Mage Enjoys a Second Lease on Life
- PSYREN
- Full Clearing Another World under a Goddess with Zero Believers
- Overgeared
- Reborn as a Space Mercenary: I Woke Up Piloting the Strongest Starship!
- HORROR COLLECTOR
- Magical Explorer

## Season 2
- The Iceblade Sorcerer Shall Rule the World
- A Returner's Magic Should Be Special
- The Detective Is Already Dead
- Saga of Tanya the Evil
- Sasaki and Peeps
- Aoashi
- Black Clover
- The Elusive Samurai

## Season 3
- The Apothecary Diaries
- As a Reincarnated Aristocrat, I'll Use My Appraisal Skill to Rise in the World

## Season 4
- Link Click

## Season 5
- The Prince of Tennis II

## Season 23
- ONE PIECE